# Hardest $k_{\text{T}}$ Summary

This is an attempt to keep all of the hardest $k_{\text{T}}$ results in one place. Scroll down a bit to find the first results.

Note: You can **zoom in on most plots by clicking on them.**

As a first step, we need to setup packages, as well as the data that we're going to use. First, the packages

In [ ]:
# Setup
import logging
from pathlib import Path
from typing import Dict, Sequence

import matplotlib
import matplotlib.pyplot as plt
from pachyderm import binned_data
import uproot
import uproot3

import jet_substructure.analysis.plot_base as pb
from jet_substructure.base import helpers, notebook_utils as nb_utils
from jet_substructure.analysis import new_plot_comparison, plot_from_skim

%load_ext autoreload
%autoreload 2

#%matplotlib inline
#%config InlineBackend.figure_formats = ["png", "pdf"]
# Don't show mpl images inline. We'll handle displaying them separately.
plt.ioff()
# Ensure the axes are legible on a dark background
matplotlib.rcParams['figure.facecolor'] = 'w'

helpers.setup_logging()
# Quiet down the matplotlib logging
logging.getLogger("matplotlib").setLevel(logging.INFO)
logging.getLogger("PIL").setLevel(logging.INFO)
logging.getLogger("pachyderm.histogram").setLevel(logging.INFO)

logger = logging.getLogger(__name__)

# General settings
embed_images = True
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)

Next, code that's likely to be shared and refactored.

## Grooming methods comparison

Setup for embedding

In [ ]:
# First, any required imports for embedding
from jet_substructure.base import skim_analysis_objects

# Next, any helper functions
def grooming_name(grooming_method: str, prefix: str) -> str:
    return f"{grooming_method}_{prefix}"

# Settings
collision_system = "embedPythia"
prefix = "hybrid"
jet_pt_bin = helpers.JetPtRange(min=40, max=120)
text_label = "Iterative splittings"
text_label += "\n" + f"${jet_pt_bin.display_str(label='')}$"
# Group some of the grooming methods together for clarity
_all_available_methods = []
_all_available_methods.extend(nb_utils.leading_kt_grooming_methods[:2])
_all_available_methods.extend(nb_utils.dynamical_grooming_methods[1:])
_all_available_methods.extend(nb_utils.soft_drop_grooming_methods[:1])
_method_groups = [
    nb_utils.leading_kt_grooming_methods[:2],
    nb_utils.dynamical_grooming_methods[1:],
    nb_utils.soft_drop_grooming_methods[:1],
    _all_available_methods
]
# Helpers for plotting responses
_matching_name_to_axis_value: Dict[str, int] = {
    "all": 0,
    "pure": 1,
    "leading_untagged_subleading_correct": 2,
    "leading_correct_subleading_untagged": 3,
    "leading_correct_subleading_mistag": 4,
    "leading_mistag_subleading_correct": 5,
    "leading_untagged_subleading_mistag": 6,
    "leading_mistag_subleading_untagged": 7,
    "swap": 8,
    "both_untagged": 9,
}
_response_types = [
    skim_analysis_objects.ResponseType(measured_like="hybrid", generator_like="det_level"),
    skim_analysis_objects.ResponseType(measured_like="hybrid", generator_like="true"),
    skim_analysis_objects.ResponseType(measured_like="det_level", generator_like="true"),
]

# Output
_output_dir = output_dir / collision_system / "RDF" / "jupyter" / prefix
_output_dir.mkdir(parents=True, exist_ok=True)
_fig_output_dir = _output_dir / "png"

# Load data for comparison
hists_embedding = {}
_successfully_loaded_methods = []
for grooming_method in nb_utils.all_grooming_methods:
    try:
        hists_embedding.update(nb_utils.load_histograms(
            filename = f"{grooming_name(grooming_method, prefix)}.root", collision_system=collision_system,
            tag = "RDF", base_path = Path("output")
        ))
        _successfully_loaded_methods.append(grooming_method)
    except FileNotFoundError:
        logger.info(f"Skipping grooming method {grooming_method} becuase the output file isn't available")
        
# Convert to boost_histogram because it simplifes projections later.
hists_embedding = {k: v.to_boost_histogram() for k, v in hists_embedding.items()}

In [ ]:
logger.info(f"\nSuccessfully loaded grooming methods: {_successfully_loaded_methods}")

In [ ]:
#plot_from_skim.compare_grooming_methods_for_substructure_prod(
#    hists=hists, grooming_methods=all_grooming_methods, prefix=prefix, output_dir=_output_dir
#)

## Embedded substructure variables

### Embedded $k_{\text{T}}$

In [ ]:
#fig, axes = plt.subplots(figsize=(24, 6), nrows=2, ncols=3, gridspec_kw={"height_ratios": [3, 1]}, sharex=True, sharey=True,)
#fig, axes = plt.subplots(figsize=(8 * 3, 6), nrows=1, ncols=3, sharey=True,)
#for methods, ax in [(leading_kt_grooming_methods[:2], axes[0]),
#                    (dynamical_grooming_methods[1:], axes[1]),
#                    (soft_drop_grooming_methods[:1], axes[2])]:
_filenames = []
for methods in _method_groups:
    _filenames.append(new_plot_comparison.plot_compare_grooming_methods_for_attribute(
        hists=hists_embedding,
        grooming_methods=methods,
        attr_name="kt",
        prefix=prefix,
        jet_pt_bin=jet_pt_bin,
        set_zero_to_nan=False,
        plot_config=pb.PlotConfig(
            name=f"kt_grooming_methods_{'_'.join(methods)}",
            panels=pb.Panel(
                axes=[
                    pb.AxisConfig("x", label=r"$k_{\text{T}}\:(\text{GeV}/c)$"),
                    pb.AxisConfig(
                        "y", label=r"$1/N_{\text{jets}}\:\text{d}N/\text{d}k_{\text{T}}\:(\text{GeV}/c)^{-1}$", log=True
                    ),
                ],
                text=pb.TextConfig(x=0.97, y=0.97, text=text_label),
                legend=pb.LegendConfig(location="lower left", font_size=14),
            ),
            figure=pb.Figure(edge_padding=dict(left=0.12, bottom=0.12)),
        ),
        output_dir=_output_dir,
        #fig=fig, ax=ax,
        plot_png=True,
    ))

In [ ]:
nb_utils.display_images([
    # First, the summary
    _filenames[-1],
    # And then the breakdown
    _filenames[:-1],
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images)

For embedded $k_{\text{T}}$, we see:

- Everything converge at high $k_{\text{T}}$.
- We see some variations between the different methods at low $k_{\text{T}}$. 
- $z > 0.2$ seems to entirely drive these differences at low $k_{\text{T}}$. It appears that SD $z > 0.2$ and leading $k_{\text{T}}$ z > 0.2 sit right on top of each other in this region.
- $z > 0.2$ methods converge back to the other methods around 5 GeV or so. They're suppressed at 1-5 GeV, but somehow are enhanced for 0-1 GeV. It seems likely that this is our shape difference.
- For dynamical grooming, dynamical time stays a bit below dynamical $k_{\text{T}}$ until they fully converge at 5. They're already close except at 0-1 GeV.

Summary: All methods seem viable at high $k_{\text{T}}$, but the shape different for $z > 0.2$ is quite apparent.

### Embedded $\Delta R$

In [ ]:
_filenames = []
for methods in _method_groups:
    _filenames.append(new_plot_comparison.plot_compare_grooming_methods_for_attribute(
        hists=hists_embedding,
        grooming_methods=methods,
        attr_name="delta_R",
        prefix=prefix,
        jet_pt_bin=jet_pt_bin,
        set_zero_to_nan=False,
        plot_config=pb.PlotConfig(
            name=f"delta_R_grooming_methods_{'_'.join(methods)}",
            panels=pb.Panel(
                axes=[
                    pb.AxisConfig("x", label=r"$\Delta R$"),
                    pb.AxisConfig("y", label=r"$1/N_{\text{jets}}\:\text{d}N/\text{d}\Delta R$", log=False),
                ],
                text=pb.TextConfig(x=0.97, y=0.97, text=text_label),
                legend=pb.LegendConfig(location="upper left", font_size=14),
            ),
            figure=pb.Figure(edge_padding=dict(left=0.12, bottom=0.12)),
        ),
        output_dir=_output_dir,
        plot_png=True,
    ))

In [ ]:
nb_utils.display_images([
    # First, the summary
    _filenames[-1],
    # And then the breakdown
    _filenames[:-1],
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images)

Observations for $\Delta R$:

- $z > 0.2$ gives a shape like roughly what we expect from the SD studies: The two peaks, with one at smal angles and one at large angles.
- Leading kt seems to be in between the dynamical grooming methods and the $z > 0.2$ methods. It peaks at large $\Delta R$ and has a rise in the small $\Delta R$ region
- Dynamical grooming has only large $\Delta R$. How much of this is background vs signal?

Summary: seems to be reasonable.

### Embedded $z$

In [ ]:
_filenames = []
for methods in _method_groups:
    _filenames.append(new_plot_comparison.plot_compare_grooming_methods_for_attribute(
        hists=hists_embedding,
        grooming_methods=methods,
        attr_name="z",
        prefix=prefix,
        jet_pt_bin=jet_pt_bin,
        set_zero_to_nan=True,
        plot_config=pb.PlotConfig(
            name=f"z_grooming_methods_{'_'.join(methods)}",
            panels=pb.Panel(
                axes=[
                    pb.AxisConfig("x", label=r"$z$"),
                    pb.AxisConfig("y", label=r"$1/N_{\text{jets}}\:\text{d}N/\text{d}z$", log=False, range=(-0.2, None)),
                ],
                text=pb.TextConfig(x=0.97, y=0.97, text=text_label),
                legend=pb.LegendConfig(location="lower right", font_size=14),
            ),
            figure=pb.Figure(edge_padding=dict(left=0.12, bottom=0.12)),
        ),
        output_dir=_output_dir,
        plot_png=True,
    ))

In [ ]:
nb_utils.display_images([
    # First, the summary
    _filenames[-1],
    # And then the breakdown
    _filenames[:-1],
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images)

Observations for $z$:

- $z > 0.2$ methods are somewhat similar, but shapes seem to be distinctly different.
- Dynamical grooming have a similar shape to those two (more simiar to leading $k_{\text{T}}$ $z > 0.2$).
- Leading $k_{\text{T}}$ is quite flat until $z < 0.1$.
- That the $z > 0.2$ curves are above the others tells us that there are cases where lower $z$ splittings have higher $k_{\text{T}}$ than some higher $z$ splittings. Best guess, this is due to low $k_{\text{T}}$ splittings.

Summary: seems to be in order, but may be useful to look at for $k_{\text{T}} > 5$. In that case, we should expect the $z > 0.2$ magnitudes to be similar for all cases.

### Embedded $n_{\text{split}}$

In [ ]:
_filenames = []
for methods in _method_groups:
    _filenames.append(new_plot_comparison.plot_compare_grooming_methods_for_attribute(
        hists=hists_embedding,
        grooming_methods=methods,
        attr_name="n_to_split",
        prefix=prefix,
        jet_pt_bin=jet_pt_bin,
        set_zero_to_nan=False,
        plot_config=pb.PlotConfig(
            name=f"number_to_split_grooming_methods_{'_'.join(methods)}",
            panels=pb.Panel(
                axes=[
                    pb.AxisConfig("x", label=r"$n_{\text{split}}$"),
                    pb.AxisConfig("y", label=r"$1/N_{\text{jets}}\:\text{d}N/\text{d}n_{\text{split}}$"),
                ],
                text=pb.TextConfig(x=0.97, y=0.97, text=text_label),
                legend=pb.LegendConfig(location="center right", font_size=14),
            ),
            figure=pb.Figure(edge_padding=dict(left=0.12, bottom=0.12)),
        ),
        output_dir=_output_dir,
        plot_png=True,
    ))

In [ ]:
nb_utils.display_images([
    # First, the summary
    _filenames[-1],
    # And then the breakdown
    _filenames[:-1],
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images)

Observations:

- All grooming methods have peaks at 1.
- $z > 0.2$ shifts to higher $n$., and seems to include a second peak (presumably from splittings that are far into the tree).
- Other methods seem to cluster together

### Embedded $n_{\text{groomed,split}}$

In [ ]:
_filenames = []
for methods in _method_groups:
    _filenames.append(new_plot_comparison.plot_compare_grooming_methods_for_attribute(
        hists=hists_embedding,
        grooming_methods=methods,
        attr_name="n_groomed_to_split",
        prefix=prefix,
        jet_pt_bin=jet_pt_bin,
        set_zero_to_nan=False,
        plot_config=pb.PlotConfig(
            name=f"number_groomed_to_split_grooming_methods_{'_'.join(methods)}",
            panels=pb.Panel(
                axes=[
                    pb.AxisConfig("x", label=r"$n_{\text{groomed,split}}$"),
                    pb.AxisConfig("y", label=r"$1/N_{\text{jets}}\:\text{d}N/\text{d}n_{\text{groomed,split}}$")
                ],
                text=pb.TextConfig(x=0.97, y=0.97, text=text_label),
                legend=pb.LegendConfig(location="center right", font_size=14),
            ),
            figure=pb.Figure(edge_padding=dict(left=0.12, bottom=0.12)),
        ),
        output_dir=_output_dir,
        plot_png=True,
    ))

In [ ]:
nb_utils.display_images([
    # First, the summary
    _filenames[-1],
    # And then the breakdown
    _filenames[:-1],
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images);

Observations:

- $z > 0.2$ means that it's almost always the first splitting (or always, for SD - see below).
- As a sanity check, SD looks right: only entries at 0 or 1, as it can't possibly have more.
- The other grooming methods seem to cluster together, with most in the first one or two splittings. However,  this is a tail.

### Embedded $n_{\text{passed grooming}}$

In [ ]:
_filenames = []
for methods in _method_groups:
    _filenames.append(new_plot_comparison.plot_compare_grooming_methods_for_attribute(
        hists=hists_embedding,
        grooming_methods=methods,
        attr_name="n_passed_grooming",
        prefix=prefix,
        jet_pt_bin=jet_pt_bin,
        set_zero_to_nan=False,
        plot_config=pb.PlotConfig(
            name=f"number_passed_grooming_grooming_methods_{'_'.join(methods)}",
            panels=pb.Panel(
                axes=[
                    pb.AxisConfig("x", label=r"$n_{\text{passed grooming}}$"),
                    pb.AxisConfig("y", label=r"$1/N_{\text{jets}}\:\text{d}N/\text{d}n_{\text{passed grooming}}$"),
                ],
                text=pb.TextConfig(x=0.97, y=0.97, text=text_label),
                legend=pb.LegendConfig(location="center right", font_size=14),
            ),
            figure=pb.Figure(edge_padding=dict(left=0.12, bottom=0.12)),
        ),
        output_dir=_output_dir,
        plot_png=True,
    ))

In [ ]:
nb_utils.display_images([
    # First, the summary
    _filenames[-1],
    # And then the breakdown
    _filenames[:-1],
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images)

Observations:

- This is equivalent to $n_{\text{SD}}$.
- Two modes of behavior, as dictated by the $z$ cut.
- All of the grooming methods of the $z$ cut sit on top of each other, as expected.

## Pb-Pb Data

Now, shift focus to the Pb-Pb data distributions

First, a bit of setup



In [ ]:
def grooming_name(grooming_method: str, prefix: str) -> str:
    return f"{grooming_method}_{prefix}"

# Settings
collision_system = "PbPb"
prefix = "data"
jet_pt_bin = helpers.JetPtRange(min=40, max=120)
text_label = "Iterative splittings"
text_label += "\n" + f"${jet_pt_bin.display_str(label='')}$"
# Group some of the grooming methods together for clarity
_all_available_methods = list(nb_utils.all_grooming_methods)
_method_groups = [
    nb_utils.leading_kt_grooming_methods,
    nb_utils.dynamical_grooming_methods,
    nb_utils.soft_drop_grooming_methods,
    _all_available_methods,
]

# Output
_output_dir = output_dir / collision_system / "RDF" / "jupyter" / prefix
_output_dir.mkdir(parents=True, exist_ok=True)
_fig_output_dir = _output_dir / "png"

# Load data for comparison
hists_data = {}
_successfully_loaded_methods = []
for grooming_method in nb_utils.all_grooming_methods:
    try:
        hists_data.update(nb_utils.load_histograms(
            filename = f"{grooming_name(grooming_method, prefix)}.root", collision_system=collision_system,
            tag = "RDF", base_path = Path("output")
        ))
        _successfully_loaded_methods.append(grooming_method)
    except FileNotFoundError:
        logger.info(f"Skipping grooming method {grooming_method} becuase the output file isn't available")
        
# Convert to boost_histogram because it simplifies projections later.
hists_data = {k: v.to_boost_histogram() for k, v in hists_data.items()}

In [ ]:
logger.info(f"\nSuccessfully loaded grooming methods: {_successfully_loaded_methods}")

### Data $k_{\text{T}}$

In [ ]:
_filenames = []
for methods in _method_groups:
    _filenames.append(new_plot_comparison.plot_compare_grooming_methods_for_attribute(
        hists=hists_data,
        grooming_methods=methods,
        attr_name="kt",
        prefix=prefix,
        jet_pt_bin=jet_pt_bin,
        set_zero_to_nan=False,
        plot_config=pb.PlotConfig(
            name=f"kt_grooming_methods_{'_'.join(methods)}",
            panels=pb.Panel(
                axes=[
                    pb.AxisConfig("x", label=r"$k_{\text{T}}\:(\text{GeV}/c)$"),
                    pb.AxisConfig(
                        "y", label=r"$1/N_{\text{jets}}\:\text{d}N/\text{d}k_{\text{T}}\:(\text{GeV}/c)^{-1}$", log=True
                    ),
                ],
                text=pb.TextConfig(x=0.97, y=0.97, text=text_label),
                legend=pb.LegendConfig(location="lower left", font_size=14),
            ),
            figure=pb.Figure(edge_padding=dict(left=0.12, bottom=0.12)),
        ),
        output_dir=_output_dir,
        plot_png=True,
    ))

In [ ]:
nb_utils.display_images([
    # First, the summary
    _filenames[-1],
    # And then the breakdown
    _filenames[:-1],
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images)

Observations:

- $k_{\text{T}}$ stats look like it might make it up to 15 GeV.
- Grooming methods are again sorted based on $z$ cut.
- For $z > 0.4$, the shape is really distorted below 5 GeV. I'm guessing that it may not be possible to unfold, but we can try.
    - Surprisingly, it seems to be suppressed all the way to high $k_{\text{T}}$.
- All dynamical grooming methods look like they should be possible to unfold.

### Data $\Delta R$

In [ ]:
_filenames = []
for methods in _method_groups:
    _filenames.append(new_plot_comparison.plot_compare_grooming_methods_for_attribute(
        hists=hists_data,
        grooming_methods=methods,
        attr_name="delta_R",
        prefix=prefix,
        jet_pt_bin=jet_pt_bin,
        set_zero_to_nan=False,
        plot_config=pb.PlotConfig(
            name=f"delta_R_grooming_methods_{'_'.join(methods)}",
            panels=pb.Panel(
                axes=[
                    pb.AxisConfig("x", label=r"$\Delta R$"),
                    pb.AxisConfig("y", label=r"$1/N_{\text{jets}}\:\text{d}N/\text{d}\Delta R$", log=False),
                ],
                text=pb.TextConfig(x=0.97, y=0.97, text=text_label),
                legend=pb.LegendConfig(location="upper left", font_size=14),
            ),
            figure=pb.Figure(edge_padding=dict(left=0.12, bottom=0.12)),
        ),
        output_dir=_output_dir,
        plot_png=True,
    ))

In [ ]:
nb_utils.display_images([
    # First, the summary
    _filenames[-1],
    # And then the breakdown
    _filenames[:-1],
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images);

Observations:

- Again, seems to be driven by $z$ cut values.
- The exception seems to be dynamical $z$, which follows the $z > 0.2$ group.
- Overall, it seems reasonable.
- There are a _ton_ of untagged for $z > 0.4$ (generally true, but most apparent here).

### Data $z$

In [ ]:
_filenames = []
for methods in _method_groups:
    _filenames.append(new_plot_comparison.plot_compare_grooming_methods_for_attribute(
        hists=hists_data,
        grooming_methods=methods,
        attr_name="z",
        prefix=prefix,
        jet_pt_bin=jet_pt_bin,
        set_zero_to_nan=True,
        plot_config=pb.PlotConfig(
            name=f"z_grooming_methods_{'_'.join(methods)}",
            panels=pb.Panel(
                axes=[
                    pb.AxisConfig("x", label=r"$z$"),
                    pb.AxisConfig("y", label=r"$1/N_{\text{jets}}\:\text{d}N/\text{d}z$", log=False, range=(-0.2, None)),
                ],
                text=pb.TextConfig(x=0.97, y=0.97, text=text_label),
                legend=pb.LegendConfig(location="upper center", font_size=14),
            ),
            figure=pb.Figure(edge_padding=dict(left=0.12, bottom=0.12)),
        ),
        output_dir=_output_dir,
        plot_png=True,
    ))

In [ ]:
nb_utils.display_images([
    # First, the summary
    _filenames[-1],
    # And then the breakdown
    _filenames[:-1],
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images)

Observations:

- Leading $k_{\text{T}}$ tends to be fairly flat, regardless of $z$ cut.
- Dynamical grooming have a similar shape to $z > 0.2$ group (more simiar to leading $k_{\text{T}}$ $z > 0.2$).
- That the $z > 0.2$ curves are above the others tells us that there are cases where lower $z$ splittings have higher $k_{\text{T}}$ than some higher $z$ splittings. Best guess, this is due to low $k_{\text{T}}$ splittings.

### Data $n_{\text{split}}$

In [ ]:
_filenames = []
for methods in _method_groups:
    _filenames.append(new_plot_comparison.plot_compare_grooming_methods_for_attribute(
        hists=hists_data,
        grooming_methods=methods,
        attr_name="n_to_split",
        prefix=prefix,
        jet_pt_bin=jet_pt_bin,
        set_zero_to_nan=False,
        plot_config=pb.PlotConfig(
            name=f"number_to_split_grooming_methods_{'_'.join(methods)}",
            panels=pb.Panel(
                axes=[
                    pb.AxisConfig("x", label=r"$n_{\text{split}}$"),
                    pb.AxisConfig("y", label=r"$1/N_{\text{jets}}\:\text{d}N/\text{d}n_{\text{split}}$"),
                ],
                text=pb.TextConfig(x=0.97, y=0.97, text=text_label),
                legend=pb.LegendConfig(location="center right", font_size=14),
            ),
            figure=pb.Figure(edge_padding=dict(left=0.12, bottom=0.12)),
        ),
        output_dir=_output_dir,
        plot_png=True,
    ))

In [ ]:
nb_utils.display_images([
    # First, the summary
    _filenames[-1],
    # And then the breakdown
    _filenames[:-1],
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images)

Observations:

- $z > 0.2$ and $z > 0.4$ perform fairly similarly, but $z > 0.4$ has a ton of untagged, so that shapes look more different than they are.
- Larger tail for dynamical $z$ compare to the other two.
- Leading $k_{\text{T}}$ and dynamical $k_{\text{T}}$ are fairly similar.


### Data $n_{\text{split,groomed}}$

In [ ]:
_filenames = []
for methods in _method_groups:
    _filenames.append(new_plot_comparison.plot_compare_grooming_methods_for_attribute(
        hists=hists_data,
        grooming_methods=methods,
        attr_name="n_groomed_to_split",
        prefix=prefix,
        jet_pt_bin=jet_pt_bin,
        set_zero_to_nan=False,
        plot_config=pb.PlotConfig(
            name=f"number_groomed_to_split_grooming_methods_{'_'.join(methods)}",
            panels=pb.Panel(
                axes=[
                    pb.AxisConfig("x", label=r"$n_{\text{groomed,split}}$"),
                    pb.AxisConfig("y", label=r"$1/N_{\text{jets}}\:\text{d}N/\text{d}n_{\text{groomed,split}}$")
                ],
                text=pb.TextConfig(x=0.97, y=0.97, text=text_label),
                legend=pb.LegendConfig(location="center right", font_size=14),
            ),
            figure=pb.Figure(edge_padding=dict(left=0.12, bottom=0.12)),
        ),
        output_dir=_output_dir,
        plot_png=True,
    ))

In [ ]:
nb_utils.display_images([
    # First, the summary
    _filenames[-1],
    # And then the breakdown
    _filenames[:-1],
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images)

Observations:

- All methods prefer the 1st splitting, with a tail for those without $z$ cuts.
- SD checks out - untagged or only the first splitting.
- Leading $k_{\text{T}}$ with a $z$ cut follows a similar trend, but for $z > 0.2$ there are a few splittings at n=2.

### Data $n_{\text{passed groomed}}$

In [ ]:
_filenames = []
for methods in _method_groups:
    _filenames.append(new_plot_comparison.plot_compare_grooming_methods_for_attribute(
        hists=hists_data,
        grooming_methods=methods,
        attr_name="n_passed_grooming",
        prefix=prefix,
        jet_pt_bin=jet_pt_bin,
        set_zero_to_nan=False,
        plot_config=pb.PlotConfig(
            name=f"number_passed_grooming_grooming_methods_{'_'.join(methods)}",
            panels=pb.Panel(
                axes=[
                    pb.AxisConfig("x", label=r"$n_{\text{passed grooming}}$"),
                    pb.AxisConfig("y", label=r"$1/N_{\text{jets}}\:\text{d}N/\text{d}n_{\text{passed grooming}}$"),
                ],
                text=pb.TextConfig(x=0.97, y=0.97, text=text_label),
                legend=pb.LegendConfig(location="upper center", font_size=14),
            ),
            figure=pb.Figure(edge_padding=dict(left=0.12, bottom=0.12)),
        ),
        output_dir=_output_dir,
        plot_png=True,
    ))

In [ ]:
nb_utils.display_images([
    # First, the summary
    _filenames[-1],
    # And then the breakdown
    _filenames[:-1],
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images)

Observations:

- Same deal as for embedded: Each value of the $z$ cut forms a class of splittings which pass the grooming.
- All grooming methods with the same $z$ cut lie on top of each other, as expected.
- Recall, this is equivalent to $n_{\text{SD}}$.

## Data vs Embedded Comparison

In order to run this section, it's required to run the setup for the embedded and data above.

### Data vs Embedded

In [ ]:
# Settings (merge with below)
jet_pt_bin = helpers.JetPtRange(min=40, max=120)
# Group some of the grooming methods together for clarity
_all_available_methods = []
_all_available_methods.extend(nb_utils.leading_kt_grooming_methods[:2])
_all_available_methods.extend(nb_utils.dynamical_grooming_methods[1:])
_all_available_methods.extend(nb_utils.soft_drop_grooming_methods[:1])
_method_groups = [
    nb_utils.leading_kt_grooming_methods[:2],
    nb_utils.dynamical_grooming_methods[1:],
    nb_utils.soft_drop_grooming_methods[:1],
    _all_available_methods
]

# Output
_output_dir = output_dir / "comparison" / "RDF" / "jupyter"
_output_dir.mkdir(parents=True, exist_ok=True)
_fig_output_dir = _output_dir / "png"

plot_from_skim.compare_grooming_methods_for_substructure_data_embed_prod(
    data_hists=hists_data,
    embed_hists=hists_embedding,
    grooming_methods=_all_available_methods,
    output_dir=_output_dir,
    rdf_plots=True,
    plot_png=True,
)

In [ ]:
def comparison_filename(grooming_method: str, variable: str, jet_pt_bin: helpers.JetPtRange, ) -> str:
    return f"{variable}_grooming_methods_{jet_pt_bin}_{grooming_method}_PbPb_hybrid_iterative_splittings"

### Data vs Embedded $k_{\text{T}}$

In [ ]:
_comparisons = []
for grooming_method in _all_available_methods:
    _comparisons.append(comparison_filename(grooming_method=grooming_method, variable="kt", jet_pt_bin=jet_pt_bin))
nb_utils.display_images([
    _comparisons[3*i:3*(i+1)] for i in range(0, int(len(_comparisons)/3) + 1)
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images)

Observations:

(Sorry, I need to fix the colors - they're automatically assigned, but got scrambled after HP)

- Embedded reproduces Pb-Pb fairly well for all grooming methods.
- At high $k_{\text{T}}$, embedded is slightly **above** Pb-Pb.
    - Different than the trends when the backgroudn subtraction was wrong.
    - Since it reproduces mid to low $k_{\text{T}}$ well, I think this is probably about right.
- Perhaps some sahpe differences.
- Embedded usually underpredicts untagged.

### $\Delta R$

In [ ]:
_comparisons = []
for grooming_method in _all_available_methods:
    _comparisons.append(comparison_filename(grooming_method=grooming_method, variable="delta_R", jet_pt_bin=jet_pt_bin))
nb_utils.display_images([
    _comparisons[3*i:3*(i+1)] for i in range(0, int(len(_comparisons)/3) + 1)
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images)

Observations:

- For all grooming methods, we have the same behavior:
    - Small angles, PbPb is enhanced.
    - Mid angles, PbPb is suppressed.
    - Large angles, PbPb is mostly in agreement, but trending towards enhanced.
- Exactly where the enhancement or supression occur depends on the grooming method.
    - ie. The details change a bit, even if the general story is the same.

### $z$

In [ ]:
_comparisons = []
for grooming_method in _all_available_methods:
    _comparisons.append(comparison_filename(grooming_method=grooming_method, variable="z", jet_pt_bin=jet_pt_bin))
nb_utils.display_images([
    _comparisons[3*i:3*(i+1)] for i in range(0, int(len(_comparisons)/3) + 1)
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images)

Observations:

- Embedded reproduces PbPb very well, with some PbPb enhancement at $z < 0.1$.
- $z_{\text{g}}$ seems to be more easily reproduced by models, etc, so perhaps this isn't so surprising.

### $n_{\text{split}}$

In [ ]:
_comparisons = []
for grooming_method in _all_available_methods:
    _comparisons.append(comparison_filename(grooming_method=grooming_method, variable="number_to_split", jet_pt_bin=jet_pt_bin))
nb_utils.display_images([
    _comparisons[3*i:3*(i+1)] for i in range(0, int(len(_comparisons)/3) + 1)
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images)

Observations:

- For all grooming methods, PbPb has more untagged (usually with large uncertainties)
- Ratio close to unity everywhere else.
- Dynamical time, dynamical $k_{\text{T}}$, and Leading $k_{\text{T}}$ are suppressed in PbPb at high $n$. Behavior disappaers for $z > 0.2$.

### $n_{\text{split,groomed}}$

In [ ]:
_comparisons = []
for grooming_method in _all_available_methods:
    _comparisons.append(comparison_filename(grooming_method=grooming_method, variable="number_groomed_to_split", jet_pt_bin=jet_pt_bin))
nb_utils.display_images([
    _comparisons[3*i:3*(i+1)] for i in range(0, int(len(_comparisons)/3) + 1)
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images)

Observations:

- Same trends as for $n_{\text{split}}$.
- Not much info for the $z > 0.2$ methods.

### $n_{\text{passed grooming}}$

In [ ]:
_comparisons = []
for grooming_method in _all_available_methods:
    _comparisons.append(comparison_filename(grooming_method=grooming_method, variable="number_passed_grooming", jet_pt_bin=jet_pt_bin))
nb_utils.display_images([
    _comparisons[3*i:3*(i+1)] for i in range(0, int(len(_comparisons)/3) + 1)
], fig_output_dir=_fig_output_dir, embed_with_base64=embed_images)

Observations:

- For the no $z$ cut methods: PbPb has enahnced untagged, and an enhancement around 2, but then agrees for high $n$.
    - I suppose PbPb must be suppresed for very large $n$ beyond the scale.
- For $z > 0.2$ methods: PbPb has enhanced untagged, suppression at mid $n$ around 2-4, and then enhanced at high $n$.
    - Note that the size of changes are actually smaller than for no $z$ cut methods.

**Overall Summary**

Embedded broadly reproduces PbPb. In the details, there are some variations. As usual, the greatest difference in behavior between grooming methods is between no $z$ cut and $z > 0.2$ methods.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

import boost_histogram as bh
for name in all_grooming_methods:
    _temp = hists.get(f"{name}_hybrid_kt", None)
    if _temp is None:
        continue
    _bh = _temp[bh.loc(40):bh.loc(120) : bh.sum, :]
    _h =  binned_data.BinnedData.from_existing_data(_bh)
    ax.errorbar(_h.axes[0].bin_centers, _h.values, yerr=_h.errors, alpha=0.7)
    ax.set_yscale("log")

In [ ]:
import numpy as np
#_h = binned_data.BinnedData.from_existing_data(hists["leading_kt_z_cut_02_hybrid_kt"][bh.loc(40):bh.loc(120):bh.sum, :])
#_h2 = binned_data.BinnedData.from_existing_data(hists["leading_kt_hybrid_kt"][bh.loc(40):bh.loc(120):bh.sum, :])
_h = binned_data.BinnedData.from_existing_data(hists["leading_kt_z_cut_02_hybrid_kt"])
_h2 = binned_data.BinnedData.from_existing_data(hists["leading_kt_hybrid_kt"])
#print(_h.values, _h2.values)
fig, ax = plt.subplots(figsize=(8,6))
mesh = ax.pcolormesh(
    _h.axes[0].bin_edges.T, _h.axes[1].bin_edges.T, _h.values.T, norm=matplotlib.colors.LogNorm(),
)

In [ ]:
import uproot
import uproot3
test_hists = {}
with uproot3.open("output/embedPythia/RDF/leading_kt_z_cut_02_hybrid.root") as f:
    for k in f.keys():
        # Remove the cycle from the name. We don't care.
        temp_k = k.decode("utf-8")
        temp_k = temp_k[: temp_k.find(";")]
        test_hists[temp_k] = f[temp_k]


In [ ]:
test_hist = test_hists["leading_kt_z_cut_02_hybrid_kt"]
(values, errors), edges = test_hist.to_numpy(flow=True, errors=True, dd=True)
#assert np.allclose(values, test_hist.values()[1:-1, 1:-1])
#test_errors = np.array(test_hist.member("fSumw2")).reshape(len(edges[0]) + 1, len(edges[1]) + 1)[1:-1, 1:-1]
#assert np.allclose(errors ** 2, test_errors)
len(edges[1])
values[:, 0]

In [ ]:
import ipyplot
ipyplot.plot_images(images = [str(_fig_output_dir / f"{filename}.png") for filename in _filenames])

In [ ]:
values

In [ ]:
filename = "output/embedPythia/RDF/leading_kt_z_cut_02_hybrid.root"
test_hist_3 = uproot3.open(filename)[b"leading_kt_z_cut_02_hybrid_kt"]
test_hist_4 = uproot.open(filename)["leading_kt_z_cut_02_hybrid_kt"]
np.isclose(test_hist_3.values, test_hist_4.values()[1:-1, 1:-1])

In [ ]:
len(edges[0]), len(edges[1]), values.shape

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
mesh = ax.pcolormesh(
    edges[1][1:-1], edges[0][1:-1], values[1:-1, 1:-1], norm=matplotlib.colors.LogNorm(),
)

## Leading $k_{\text{T}}$ Responses

Show the general responses...

### Subjet (Prong) Matching Studies

#### Upper Left Corner Studies

Main source appears to be:
- Losing the leading subjet, and then the subleading -> leading. The subleading is then paired with a random constituent.
- ...

What happens to the subjets when they are lost? Where do they end up?

In [ ]:
# Leading...

In [ ]:
# Subleaindg...

Summary:

...

## Unfolding

Shown below for dynamical $k_{\text{T}}$:

In [ ]:
import seaborn as sns

from jet_substructure.analysis import plot_unfolding
from jet_substructure.analysis import plot_base as pb

First, the binning:

In [ ]:
# Extract binning from the histograms...
print(f"True kt binning: {}")
print(f"True pt binning: {}")
print(f"Smeared kt binning: {}")
print(f"Smeared pt binning: {}")

In [ ]:
# Setup our input object
collision_system = "PbPb"
input_file = plot_unfolding.InputFile(
    "kt",
    "dynamical_kt",
    smeared_var_range=helpers.KtRange(3, 15),
    smeared_untagged_var=helpers.KtRange(2, 3),
    smeared_pt_range=helpers.JetPtRange(40, 120),
    n_iter_compare=3,
    suffix="broadTrueBins",
)
hists, output_dir = plot_unfolding.setup(input_file=input_file, collision_system=collision_system)
tag = ""

In [ ]:
reload(plot_unfolding)

In [ ]:
%%html

<style>
    .jp-OutputArea-child {
        display: inline-block;
    }
</style>

In [ ]:
figures = []
with sns.color_palette("Paired", n_colors=input_file.max_iter):
    n_iter_for_ratio = input_file.n_iter_compare
    jet_pt_for_text = helpers.RangeSelector(60, 80)
    text = f"${jet_pt_for_text.display_str(label='true')}$"
    figures.append(plot_unfolding.plot_unfolded(
        hists=hists,
        projection_func=plot_unfolding._project_kt,
        efficiency_func=plot_unfolding._efficiency_kt,
        n_iter_for_ratio=n_iter_for_ratio,
        max_iter=input_file.max_iter,
        true_bin=helpers.RangeSelector(60, 80),
        tag=tag,
        plot_config=pb.PlotConfig(
            name=f"unfolded_{input_file.substructure_variable}_true_pt_60_80",
            panels=[
                # Main panel
                pb.Panel(
                    axes=[
                        pb.AxisConfig(
                            "y",
                            label=fr"$\text{{d}}N/\text{{d}}k_{{\text{{T}}}}\:(\text{{GeV}}/c)^{{-1}}$",
                            log=True,
                        )
                    ],
                    legend=pb.LegendConfig(location="lower left"),
                    text=pb.TextConfig(text, 0.97, 0.97),
                ),
                # Ratio
                pb.Panel(
                    axes=[
                        pb.AxisConfig(
                            "y",
                            label=fr"Ratio to iter {n_iter_for_ratio}"
                            if n_iter_for_ratio > 0
                            else "Ratio to true",
                            range=(0.5, 1.5),
                        ),
                    ],
                ),
                pb.Panel(
                    axes=[
                        pb.AxisConfig("x", label=r"$k_{\text{T}}\:(\text{GeV}/c)$", range=(-0.5, 15)),
                        pb.AxisConfig("y", label="Ratio to true", range=(0.5, 1.5),),
                    ],
                ),
            ],
            figure=pb.Figure(edge_padding=dict(bottom=0.06)),
        ),
        output_dir=output_dir,
    ))
    # 40-120 true pt.
    jet_pt_for_text = helpers.RangeSelector(40, 120)
    text = f"${jet_pt_for_text.display_str(label='true')}$"
    figures.append(plot_unfolding.plot_unfolded(
        hists=hists,
        projection_func=plot_unfolding._project_kt,
        efficiency_func=plot_unfolding._efficiency_kt,
        n_iter_for_ratio=n_iter_for_ratio,
        max_iter=input_file.max_iter,
        true_bin=helpers.RangeSelector(40, 120),
        tag=tag,
        plot_config=pb.PlotConfig(
            name=f"unfolded_{input_file.substructure_variable}_true_pt_40_120",
            panels=[
                # Main panel
                pb.Panel(
                    axes=[
                        pb.AxisConfig(
                            "y",
                            label=fr"$\text{{d}}N/\text{{d}}k_{{\text{{T}}}}\:(\text{{GeV}}/c)^{{-1}}$",
                            log=True,
                        )
                    ],
                    legend=pb.LegendConfig(location="lower left"),
                    text=pb.TextConfig(text, 0.97, 0.97),
                ),
                # Ratio
                pb.Panel(
                    axes=[
                        pb.AxisConfig(
                            "y",
                            label=fr"Ratio to iter {n_iter_for_ratio}"
                            if n_iter_for_ratio > 0
                            else "Ratio to true",
                            range=(0.5, 1.5),
                        ),
                    ],
                ),
                pb.Panel(
                    axes=[
                        pb.AxisConfig("x", label=r"$k_{\text{T}}\:(\text{GeV}/c)$", range=(-0.5, 15)),
                        pb.AxisConfig("y", label="Ratio to true", range=(0.5, 1.5),),
                    ],
                ),
            ],
        ),
        output_dir=output_dir,
    ))
    text = ""
    figures.append(plot_unfolding.plot_unfolded(
        hists=hists,
        projection_func=plot_unfolding._project_pt,
        efficiency_func=plot_unfolding._efficiency_pt,
        n_iter_for_ratio=n_iter_for_ratio,
        max_iter=input_file.max_iter,
        true_bin=helpers.RangeSelector(0, 25),
        tag=tag,
        plot_config=pb.PlotConfig(
            name="unfolded_pt",
            panels=[
                # Main panel
                pb.Panel(
                    axes=[
                        pb.AxisConfig(
                            "y", label=r"$\text{d}N/\text{d}p_{\text{T}}\:(\text{GeV}/c)^{-1}$", log=True
                        )
                    ],
                    legend=pb.LegendConfig(location="lower left"),
                    text=pb.TextConfig(text, 0.97, 0.97),
                ),
                # Ratio
                pb.Panel(
                    axes=[
                        pb.AxisConfig(
                            "y",
                            label=fr"Ratio to iter {n_iter_for_ratio}"
                            if n_iter_for_ratio > 0
                            else "Ratio to true",
                            range=(0.5, 1.5),
                        ),
                    ],
                ),
                pb.Panel(
                    axes=[
                        pb.AxisConfig("x", label=r"$p_{\text{T}}\:(\text{GeV}/c)$"),
                        pb.AxisConfig("y", label="Ratio to true", range=(0.5, 1.5),),
                    ],
                ),
            ],
        ),
        output_dir=output_dir,
    ))
figures[0]

In [ ]:
import matplotlib.pyplot as plt
plt.show(figures[0])